In [1]:
import os

# Update you config directory as needed
os.environ["PSYNC_CONFIG_DIR"] = "../../../config"

In [2]:
import logging

logging.basicConfig(
    format="%(asctime)s,%(msecs)03d %(levelname)-8s [%(filename)s:%(lineno)d] %(message)s",
    datefmt="%Y-%m-%dT%H:%M:%S",
    level=logging.DEBUG,
)

# Collections

The Tidal service implements the standard collection interfaces that are common across all `plistsync` services. By inheriting from the base classes and conforming to the established protocols, Tidal provides a unified interface that works consistently with other services.

## Library Collection

The {py:class}`TidalLibraryCollection <plistsync.services.tidal.TidalLibraryCollection>` represents the full Tidal catalog from the perspective of the authenticated user. 

In [3]:
from plistsync.services.tidal import TidalLibraryCollection

library = TidalLibraryCollection()

2026-02-04T21:30:31,053 INFO     [file.py:127] Loading config file: /mnt/Repositories/plistsync/config/config.yml
2026-02-04T21:30:31,153 INFO     [file.py:127] Loading config file: /mnt/Repositories/plistsync/config/config.yml


### Track Lookup

The Tidal library collections implements the {py:class}`GlobalLookup <plistsync.core.collection.GlobalLookup>` protocol, allowing you to search for tracks across Tidal's entire music universe by their ids. Specifically, tidal supports lookups by `isrc` and `tidal_id`.

In [4]:
# By tidal id
if track1 := library.find_by_global_ids({"isrc": "GBBZH9601601"}):
    print(track1)

# By isrc
if track2 := library.find_by_global_ids({"tidal_id": "239951925"}):
    print(track2)

# TODO: This is broken somehow
assert track1 == track2

2026-02-04T21:30:31,530 DEBUG    [connectionpool.py:1049] Starting new HTTPS connection (1): openapi.tidal.com:443
2026-02-04T21:30:31,867 DEBUG    [connectionpool.py:544] https://openapi.tidal.com:443 "GET /v2/tracks?filter%5Bisrc%5D=GBBZH9601601&include=albums&include=artists HTTP/1.1" 200 None
2026-02-04T21:30:32,021 DEBUG    [connectionpool.py:544] https://openapi.tidal.com:443 "GET /v2/tracks?filter%5Bid%5D=239951925&include=albums&include=artists HTTP/1.1" 200 None


Track[Origin Unknown > Valley Of The Shadows, hash: -8292429380783153179]
Track[Origin Unknown > Valley Of The Shadows, hash: -4790946608272251374]


AssertionError: 

If you need/want to lookup many tracks at once, consider using {py:meth}`TidalLibraryCollection.find_many_by_global_ids <plistsync.services.tidal.TidalLibraryCollection.find_many_by_global_ids>` for better performance.

## Playlist Collection

The {py:class}`TidalLibraryCollection <plistsync.services.tidal.TidalLibraryCollection>` represents a playlist than is accessible by the currently authenticated user. 

### Retrieving playlists

You can retrieve all playlists of a user using the library's {py:meth}`TidalLibraryCollection.playlists <plistsync.services.tidal.TidalLibraryCollection.playlists>` property.


In [5]:
playlists = library.playlists
for playlist in playlists:
    print(f"Playlist: {playlist.name} ({len(playlist)} tracks)")

2026-02-04T21:30:34,224 DEBUG    [connectionpool.py:544] https://openapi.tidal.com:443 "GET /v2/users/me HTTP/1.1" 200 None
2026-02-04T21:30:34,372 DEBUG    [connectionpool.py:544] https://openapi.tidal.com:443 "GET /v2/playlists?filter%5Bowners.id%5D=197912502 HTTP/1.1" 200 None
2026-02-04T21:30:34,519 DEBUG    [connectionpool.py:544] https://openapi.tidal.com:443 "GET /v2/playlists?filter%5Bowners.id%5D=197912502&page%5Bcursor%5D=eyJpZCI6ImUxZThjODNlLTQ1OTUtNDQzYS04Y2ViLTlkOTQxMzdmZjEyMyIsIm5hbWUiOiJIbW0iLCJhZGRlZEF0IjoiMjAyNC0wOC0yMlQxNDoxMjoxOS4wMzMzMDdaIiwidXBkYXRlZEF0IjoiMjAyNC0wOS0yNVQxMDo0MzoyNi44MjJaIiwiaXRlbVR5cGUiOiJVU0VSX1BMQVlMSVNUIiwiYXJ0aXN0TmFtZSI6bnVsbCwicmVsZWFzZWRBdCI6bnVsbCwibWl4VHlwZSI6bnVsbH0%3D&filter%5Bowners.id%5D=197912502 HTTP/1.1" 200 None


Playlist: DNB Copy (2 tracks)
Playlist: DNB Copy (1 tracks)
Playlist: DNB Copy (3 tracks)
Playlist: DNB Copy (0 tracks)
Playlist: DNB Copy (0 tracks)
Playlist: My New Playlist (0 tracks)
Playlist: Melodic Techno (2 tracks)
Playlist: funky (3 tracks)
Playlist: HipHop Bass  (4 tracks)
Playlist: Jungle (1 tracks)
Playlist: House (10 tracks)
Playlist: ? (2 tracks)
Playlist: Drum and Bass Top 100 (130 tracks)
Playlist: Nele Workshop (9 tracks)
Playlist: More than 20 tracks! (49 tracks)
Playlist: Disco sunny (14 tracks)
Playlist: A bit of liquid (2 tracks)
Playlist: New dnb (9 tracks)
Playlist: house (6 tracks)
Playlist: Hmm (17 tracks)
Playlist: Relaxed get together (disco?) (256 tracks)
Playlist: GTown prep (43 tracks)
Playlist: Tribal dnb (68 tracks)
Playlist: Transition (3 tracks)
Playlist: Burg Rave Prep (42 tracks)
Playlist: Tribal (3 tracks)
Playlist: Dirty DnB (15 tracks)
Playlist: Litt DnB 22|23 (101 tracks)
Playlist: Litt DnB 19 (142 tracks)
Playlist: TEST (1 tracks)
Playlist: burg

If you just want a specific playlist, you can use the library's {py:meth}`TidalLibraryCollection.get_playlist <plistsync.services.tidal.TidalLibraryCollection.get_playlist>` method to retrieve a playlist.

In [6]:
if pl := library.get_playlist(
    name="Drum and Bass Top 100"
    # id = "5e9f3d7e-325d-4cbf-afde-9538b0aa8819"
    # url = "https://tidal.com/playlist/5e9f3d7e-325d-4cbf-afde-9538b0aa8819"
):
    print(f"Found playlist: {pl.name} ({len(pl)} tracks)")
else:
    print("Playlist not found.")

2026-02-04T21:30:34,836 DEBUG    [connectionpool.py:544] https://openapi.tidal.com:443 "GET /v2/users/me HTTP/1.1" 200 None
2026-02-04T21:30:34,992 DEBUG    [connectionpool.py:544] https://openapi.tidal.com:443 "GET /v2/playlists?filter%5Bowners.id%5D=197912502 HTTP/1.1" 200 None
2026-02-04T21:30:35,136 DEBUG    [connectionpool.py:544] https://openapi.tidal.com:443 "GET /v2/playlists?filter%5Bowners.id%5D=197912502&page%5Bcursor%5D=eyJpZCI6ImUxZThjODNlLTQ1OTUtNDQzYS04Y2ViLTlkOTQxMzdmZjEyMyIsIm5hbWUiOiJIbW0iLCJhZGRlZEF0IjoiMjAyNC0wOC0yMlQxNDoxMjoxOS4wMzMzMDdaIiwidXBkYXRlZEF0IjoiMjAyNC0wOS0yNVQxMDo0MzoyNi44MjJaIiwiaXRlbVR5cGUiOiJVU0VSX1BMQVlMSVNUIiwiYXJ0aXN0TmFtZSI6bnVsbCwicmVsZWFzZWRBdCI6bnVsbCwibWl4VHlwZSI6bnVsbH0%3D&filter%5Bowners.id%5D=197912502 HTTP/1.1" 200 None
2026-02-04T21:30:35,281 DEBUG    [connectionpool.py:544] https://openapi.tidal.com:443 "GET /v2/playlists?filter%5Bid%5D=5e9f3d7e-325d-4cbf-afde-9538b0aa8819 HTTP/1.1" 200 None


Found playlist: Drum and Bass Top 100 (130 tracks)


:::{note}
This method supports lookup by various identifiers, including ``name=``, ``id=``, or ``url=``.
Lookup by name will return ``None`` if no matching playlist is found, while lookups by other identifiers will raise a ``ValueError`` if the playlist cannot be resolved.
:::

### Creating playlists

Currently we do not support creating new playlists with a unified interface. We will reevaluate this at a later point in time. For now you need to use the Tidal API directly to create new playlists.

In [7]:
from plistsync.services.tidal import TidalPlaylistCollection

resource, lookup = library.api.playlist.create(
    name="DNB Copy",
    description="Created via plistsync",
)
pl = TidalPlaylistCollection.from_response_data(library, resource, lookup)


2026-02-04T21:30:36,867 DEBUG    [connectionpool.py:544] https://openapi.tidal.com:443 "POST /v2/playlists HTTP/1.1" 201 None


### Updating a playlist

For updating a playlist, you should use the playlist's {py:meth}`TidalPlaylistCollection.edit <plistsync.services.tidal.TidalPlaylistCollection.edit>` context manager. This ensures that all changes are properly saved back to Tidal when you exit the context. This also minifies changes and therefore reduces API calls.


In [8]:
# Updating a playlist description/name
with pl.edit():
    pl.description = "Updated Description"

pl.description

2026-02-04T21:30:38,187 DEBUG    [connectionpool.py:544] https://openapi.tidal.com:443 "PATCH /v2/playlists/72508cd6-1878-4af2-86cb-d424f34ecc36 HTTP/1.1" 204 0
2026-02-04T21:30:38,335 DEBUG    [connectionpool.py:544] https://openapi.tidal.com:443 "GET /v2/playlists/72508cd6-1878-4af2-86cb-d424f34ecc36/relationships/items?include=items&include=items.albums&include=items.artists HTTP/1.1" 200 None


'Updated Description'

In [9]:
from plistsync.services.tidal import TidalPlaylistCollection

new_track = library.find_by_global_ids({"isrc": "GBBZH9601601"})
assert new_track is not None

with pl.edit():
    # TODO: We should reevaluate the type hierarchy here.
    # for now, we have to manually cast TidalTrack to TidalPlaylistCollection.
    # this will become more convenient, eventually.
    pl.tracks.append(new_track)
    pl.tracks.append(new_track)


2026-02-04T21:30:38,798 DEBUG    [connectionpool.py:544] https://openapi.tidal.com:443 "GET /v2/tracks?filter%5Bisrc%5D=GBBZH9601601&include=albums&include=artists HTTP/1.1" 200 None
2026-02-04T21:30:39,628 DEBUG    [connectionpool.py:544] https://openapi.tidal.com:443 "POST /v2/playlists/72508cd6-1878-4af2-86cb-d424f34ecc36/relationships/items HTTP/1.1" 201 0
2026-02-04T21:30:40,008 DEBUG    [connectionpool.py:544] https://openapi.tidal.com:443 "POST /v2/playlists/72508cd6-1878-4af2-86cb-d424f34ecc36/relationships/items HTTP/1.1" 201 0
2026-02-04T21:30:40,183 DEBUG    [connectionpool.py:544] https://openapi.tidal.com:443 "GET /v2/playlists/72508cd6-1878-4af2-86cb-d424f34ecc36/relationships/items?include=items&include=items.albums&include=items.artists HTTP/1.1" 200 None


In [ ]:
with pl.edit():
    pl.tracks.pop()


2026-02-04T21:30:48,971 DEBUG    [connectionpool.py:544] https://openapi.tidal.com:443 "DELETE /v2/playlists/72508cd6-1878-4af2-86cb-d424f34ecc36/relationships/items HTTP/1.1" 204 0
2026-02-04T21:30:49,154 DEBUG    [connectionpool.py:544] https://openapi.tidal.com:443 "GET /v2/playlists/72508cd6-1878-4af2-86cb-d424f34ecc36/relationships/items?include=items&include=items.albums&include=items.artists HTTP/1.1" 200 None


In [ ]:
with pl.edit():
    pl.tracks.insert(0, pl.tracks.pop())